# Analysis Template: Binding

Outlines a simple approach for fitting $K_{d}$ values to on-chip binding datasets.

In [ ]:
#enables autoreloding of modules
%load_ext autoreload
%autoreload 2

from htbam_analysis.db_api.htbam_db_api import LocalHtbamDBAPI
from htbam_analysis.analysis.experiment import HTBAMExperiment
from htbam_analysis.analysis import plot
from htbam_analysis.db_api.units import units
from pathlib import Path

#enable inline plotting of matplotlib figures
%matplotlib inline

#set the figure format to SVG
%config InlineBackend.figure_format = 'svg'

## 1. Connect DB Api

First, we load your data. Make sure to use the correct path, filenames, and column names for standard and kinetics data.

In [ ]:
### PARAMETERS:
EGFP_SLOPE = 91900.03 / 9 * (units.RFU / units.nM)

# Print our current working directory, for convenience:
print('Currently in: ', Path.cwd())

# Make sure to use the correct relative path!
root = '/path/to/root/' 


db_conn = LocalHtbamDBAPI()

# load button quant data
db_conn.load_run(
    run_name = 'button_quant',
    csv_path = root + "button_quant.csv",
    run_type = "button_quant",
)

# load binding data
db_conn.load_run(
    run_name = "binding", 
    csv_path = root + "binding.csv.bz2", 
    run_type = "binding",
    conc_unit = units.uM, 
    time_unit = units.s,
    concentration_col = "prey_concentration"
    )

htbam_experiment = HTBAMExperiment(db_conn)

## 2. Enzyme Quant

In [ ]:
from htbam_analysis.analysis.transform import transform_data

button_concentrations = transform_data(
    data_objs = [htbam_experiment.get_run('button_quant')],              # e.g. a Data2D or Data3D or Data4D instance
    expr=f'(a.luminance / slope)',             # e.g. "(luminance - intercept) / slope"
    expression_vars={'slope': EGFP_SLOPE},
    output_name='concentration'       # name of the new field, e.g. "concentration"
)

htbam_experiment.set_run('enzyme_concentrations', button_concentrations)     # save the fit results

In [ ]:
plot.plot_enzyme_concentration_chip(htbam_experiment, analysis_name='enzyme_concentrations')

## 3. Filter data

Before fitting, we apply masks to filter our data 

In [ ]:
# These are the "Masks" we use to select which data we want.

# In this example we use:
# expression_threshold = 1.0nM,
# positive_initial_slope_filter = True

from htbam_analysis.analysis.filter import filter_expression_cutoff, filter_fluorescence_ratio, filter_initial_rates_r2_cutoff, filter_standard_curve_r2_cutoff

binding_data = htbam_experiment.get_run("binding")       # retrieve our raw data

# Fluorescence ratio
fluorescence_ratio_mask =   filter_fluorescence_ratio(binding_data, min_ratio=0, max_ratio=10)          # positive slope

# Expression:
expression_mask =               filter_expression_cutoff(button_concentrations, binding_data, expression_cutoff=1) # 1 nM

# Save the masks to the experiment
htbam_experiment.set_run('fluorescence_ratio_mask', fluorescence_ratio_mask)
htbam_experiment.set_run('expression_mask', expression_mask)

Before we apply the masks, let's look at them to see what's making it through. If we're not happy, we can go up a cell and adjust the parameters.

In [ ]:
# Plot the masks (Do one at a time)
# plot.plot_mask_chip(htbam_experiment, mask_name='expression_mask')
# plot.plot_mask_chip(htbam_experiment, mask_name='fluorescence_ratio_mask')

Finally, let's apply the masks:

In [ ]:
# Apply the masks and save the results
htbam_experiment.apply_mask(run_name='binding', 
                            dep_variables = ['fluorescence_ratio'], 
                            save_as = 'binding_masked',
                            mask_names = ['expression_mask', 'fluorescence_ratio_mask'])


Need to manually mask out more points? Use our GUI to select them.

In [ ]:
# from htbam_analysis.analysis.interactive import ImageMaskPicker

# # You can pass multiple images to switch between them!
# # Only works with "summary_image" outputs from processing.
# images = {
#     'Binding Image': './path/to/your/image.tif',
# }

# picker = ImageMaskPicker(images, n_cols=32, n_rows=56)
# picker.show()

When you're done, make sure to apply the new mask:

In [ ]:
# manual_qc_mask = picker.get_mask(binding_data, mask_name='manual_qc_mask')
# htbam_experiment.set_run('manual_qc_mask', manual_qc_mask)
# #plot.plot_mask_chip(htbam_experiment, mask_name='manual_qc_mask')

# htbam_experiment.apply_mask(run_name='binding_masked', 
#                             dep_variables = ['fluorescence_ratio'],
#                             mask_names=['manual_qc_mask'],
#                             save_as = 'binding_masked') # Overwrite the previous masked data

Let's delete chambers that don't have enough concentrations that passed filtering:

In [ ]:
# Now, we'll delete any wells that have fewer than 5 initial rates:
from htbam_analysis.analysis.filter import filter_number_concentrations

binding_data = htbam_experiment.get_run('binding_masked')       # retrieve our raw data
number_points_mask = filter_number_concentrations(binding_data, min_concentrations=5, var_to_check='fluorescence_ratio')
htbam_experiment.set_run('number_points_mask', number_points_mask)
htbam_experiment.apply_mask(run_name='binding_masked', 
                            dep_variables = ['fluorescence_ratio'],
                            mask_names=['number_points_mask'],
                            save_as = 'binding_masked') # Overwrite the previous masked data

## 4. Fit binding isotherms

In [ ]:
from htbam_analysis.analysis.fit import fit_binding_isotherm

# fit binding isotherm for each chamber
binding_experiment_data = htbam_experiment.get_run("binding_masked")
binding_fits, binding_preds, binding_mask = fit_binding_isotherm(
    data = binding_experiment_data,
    concentrations_to_fit = {
        c.magnitude: True for c in binding_experiment_data.indep_vars.concentration
        } # optionally mask bad datapoints
    )
htbam_experiment.set_run("binding_fits", binding_fits)
htbam_experiment.set_run("binding_preds", binding_preds)

We've fit our binding isotherms. Now, let's filter out poor fits.

First, we filter by R2, then we make sure we have enough replicates.

In [ ]:
from htbam_analysis.analysis.filter import filter_r2_cutoff, filter_number_replicates
 
binding_fits = htbam_experiment.get_run('binding_fits')

# Change your R2 cutoff!
binding_fits_mask = filter_r2_cutoff(binding_fits, r2_cutoff=0.8) 
binding_fits_replicate_mask = filter_number_replicates(binding_fits, min_replicates=5, var_to_check='Kd')

htbam_experiment.set_run('binding_fits_mask', binding_fits_mask)  # save the fit results
htbam_experiment.set_run('binding_fits_replicate_mask', binding_fits_replicate_mask)

# plot.plot_mask_chip(experiment=htbam_experiment, mask_name='binding_fits_mask')

Once we have a good filter, we apply the mask to the data and plot:

In [ ]:
# Apply the mask to the fits and save the results:
htbam_experiment.apply_mask(run_name='binding_fits', 
                            dep_variables = ['Kd', 'r_max', 'r_squared'],
                            save_as = 'binding_fits_masked',
                            mask_names = ['binding_fits_mask', 'binding_fits_replicate_mask'])

In [ ]:
plot.plot_binding_isotherm_chip(htbam_experiment, 'binding_masked', 'binding_fits_masked', 'binding_preds')

To fit samples where binding is not saturated, we can specify a global $r_{max}$ parameter, estimated from fits from tight binders that are saturated under assay conditions.

In [ ]:
from htbam_analysis.analysis.fit import fit_binding_isotherm

# first, fit binding isotherm for each chamber with a global r_max parameter
# here, we estimate this parameter from the tight binders that are saturated under assay conditions
binding_experiment_data = htbam_experiment.get_run("binding_masked")
binding_fixed_rmax_fits, binding_fixed_rmax_preds, binding_fixed_rmax_mask = fit_binding_isotherm(
    data = binding_experiment_data,
    concentrations_to_fit = {
        c.magnitude: True for c in binding_experiment_data.indep_vars.concentration
        },

    tight_binder_ids = ['WT'],

    # alternatively, you can specify a global r_max parameter
    # r_max = 3

    )
htbam_experiment.set_run("binding_fixed_rmax_fits", binding_fixed_rmax_fits)
htbam_experiment.set_run("binding_fixed_rmax_preds", binding_fixed_rmax_preds)

# second, filter the fits in a similar fashion as above
# Change your R2 cutoff!
binding_fits_fixed_rmax_mask = filter_r2_cutoff(binding_fixed_rmax_fits, r2_cutoff=0.8) 
binding_fits_fixed_rmaxreplicate_mask = filter_number_replicates(binding_fixed_rmax_fits, min_replicates=5, var_to_check='Kd')

htbam_experiment.set_run('binding_fits_fixed_rmax_mask', binding_fits_fixed_rmax_mask)  # save the fit results
htbam_experiment.set_run('binding_fits_fixed_rmax_replicate_mask', binding_fits_fixed_rmaxreplicate_mask)

# apply the mask to the fits and save the results:
htbam_experiment.apply_mask(run_name='binding_fixed_rmax_fits', 
                            dep_variables = ['Kd', 'r_max', 'r_squared'],
                            save_as = 'binding_fixed_rmax_fits_masked',
                            mask_names = ['binding_fits_fixed_rmax_mask', 'binding_fits_fixed_rmax_replicate_mask'])

In [ ]:
plot.plot_binding_isotherm_chip(htbam_experiment, 'binding_masked', 'binding_fixed_rmax_fits_masked', 'binding_fixed_rmax_preds')

Filter outliers using a Z-score filter.

In [ ]:
# NaN all chamber values where K_m is 1.5 standard deviations away from the mean
z_score_threshold = 1.5

# First, we make a boolean mask of all chambers that are within 3 standard deviations of the mean K_m across the entire device.
outlier_mask_kd = transform_data(
    data_objs = [htbam_experiment.get_run('binding_fixed_rmax_fits_masked')], # or use masked_fits_with_kcat_filtered
    expr=f'~(np.abs(a.chamber.Kd - np.nanmean(a.sample.Kd)) > z_score_threshold * np.nanstd(a.sample.Kd))',
    expression_vars={'z_score_threshold': z_score_threshold},
    output_name='mask'
)

outlier_mask_expression = transform_data(
    data_objs = [htbam_experiment.get_run('enzyme_concentrations')], # or use masked_fits_with_kcat_filtered
    expr=f'~(np.abs(a.chamber.concentration - np.nanmean(a.sample.concentration)) > z_score_threshold * np.nanstd(a.sample.concentration))',
    expression_vars={'z_score_threshold': z_score_threshold},
    output_name='mask'
)

# And them all together:
outlier_mask = transform_data(
    data_objs = [outlier_mask_kd, outlier_mask_expression],
    expr=f'a.mask.magnitude & b.mask.magnitude',
    output_name='mask'
)

htbam_experiment.set_run('outlier_mask', outlier_mask)
plot.plot_mask_chip(htbam_experiment, mask_name='outlier_mask')

# Apply the mask to the fits and save the results:
htbam_experiment.apply_mask(run_name='binding_fixed_rmax_fits_masked',
                            dep_variables = ['Kd', 'r_max', 'r_squared'], 
                            save_as = 'binding_fixed_rmax_fits_masked_filtered',
                            mask_names = ['outlier_mask'])

                            

## 7. Export to CSV

All of our data is sent to CSV for further processing.

In [ ]:
htbam_experiment.export_binding_chamber_data(run_name='binding_fixed_rmax_fits_masked_filtered', enzyme_concentration_run_name='enzyme_concentrations', file_name='binding_chamber_data.csv')
htbam_experiment.export_binding_sample_data(run_name='binding_fixed_rmax_fits_masked_filtered', enzyme_concentration_run_name='enzyme_concentrations', file_name='binding_sample_data.csv')

Next, we output a PDF with raw data and fitted binding isotherms:

In [ ]:
htbam_experiment.export_binding_subplots_by_sample(
                           analysis_name='binding_masked',
                           model_fit_name='binding_fixed_rmax_fits_masked_filtered',
                           dep_var_label='fluorescence_ratio',
                           export_path = 'binding_plots.pdf')